<a href="https://colab.research.google.com/github/Ashu-Shukla-1309/supreme-goggles/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ashu-Shukla-1309/supreme-goggles/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Task Type: Regression and Opportunity Scoring / Ranking (Learning to Rank)

Primary Task: Regression — Training an expected-value model to predict a page's expected organic performance (e.g., expected impressions_90d or baseline ctr) based on its ranking position, content characteristics, and age.Derived Task:

Scoring & Ranking — Computing an Opportunity Score defined as:$$\text{Opportunity Score} = \text{Expected Traffic} - \text{Actual Traffic}$$Pages are then sorted in descending order of this residual gap to generate a prioritized action playbook for content refreshes.

In [9]:
import pandas as pd
import numpy as np

# Quick check on the target distribution task
print("ML Task Definition:")
print("- Model Type: Regressor (Random Forest / Gradient Boosting)")
print("- Score Type: Expected Value Residual (Opportunity Gap)")
print("- Output: Ranked list of content refresh candidates")

ML Task Definition:
- Model Type: Regressor (Random Forest / Gradient Boosting)
- Score Type: Expected Value Residual (Opportunity Gap)
- Output: Ranked list of content refresh candidates


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Predicted Proxy / Target: impressions_90d (or expected ctr) predicted exclusively from non-leaky, pre-decision features such as avg_position, word_count, content_age_days, and days_since_last_update.Derived Target Label: $\text{Residual Gap} = \hat{y}_{\text{expected}} - y_{\text{actual}}$

Origin of the Label: This is a derived proxy. Actual performance metrics ($y_{\text{actual}}$) are observed directly from search engine console data, while expected performance ($\hat{y}_{\text{expected}}$) is generated by fitting our regression model on healthy historical benchmarks. A large positive residual indicates a page severely underperforming its positional benchmark.

In [10]:
# Verify target variable availability and proxy calculation concept
proxy_features = ["avg_position", "word_count", "days_since_last_update", "content_age_days"]
target_col = "impressions_90d"

print(f"Pre-decision features for expected value model: {proxy_features}")
print(f"Observed target variable: {target_col}")

Pre-decision features for expected value model: ['avg_position', 'word_count', 'days_since_last_update', 'content_age_days']
Observed target variable: impressions_90d


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Primary Evaluation Metric:

-Precision@K (specifically Precision@20 and Precision@50). Of the top $K$ pages flagged by our Opportunity Score, what fraction are actually declining in traffic (trend_direction == 'down') or suffering severe CTR collapse?

Secondary Metrics:

-NDCG@K (Normalized Discounted Cumulative Gain): Measures whether the pages with the largest recoverable loss appear at the very top of the ranked list.

-MAE / RMSE: Evaluates the fit quality of the underlying expected-value regressor.

What 'Good' Looks Like: Achieving a Precision@50 of >0.60 (60%+), significantly outperforming the ~49% baseline decay rate in the dataset when tested on client-holdout validation splits.

In [11]:
def precision_at_k(scores, labels, k=50):
    """Calculates Precision@K for a ranked score list."""
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return float(topk.mean())

print("Precision@K evaluation metric initialized.")

Precision@K evaluation metric initialized.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of Analysis: A single unique content item (URL/page) for a given client over a historical performance period.

Granularity: One row = One unique content page (content_hash_id).

In [12]:
import os, sys, subprocess
import pandas as pd
import numpy as np

# 1. Setup repository path in Colab
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    if os.path.basename(os.getcwd()) != REPO_DIR:
        os.chdir(REPO_DIR)

# 2. Load dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# 3. Define target label (1 = declining, 0 = stable/growing)
df["is_declining"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Display Unit of Analysis (Removed content_hash_id)
unit_of_analysis = df[["avg_position", "ctr", "impressions_90d", "days_since_last_update", "is_declining"]]
print(f"Dataframe Shape: {unit_of_analysis.shape[0]} rows (pages) x {unit_of_analysis.shape[1]} columns")
print("Unit of Analysis (1 row = 1 anonymized content page):")
unit_of_analysis.head(5)

Dataframe Shape: 30000 rows (pages) x 5 columns
Unit of Analysis (1 row = 1 anonymized content page):


,avg_position,ctr,impressions_90d,days_since_last_update,is_declining
0,10.6,0.76,3803,20,1
1,20.3,0.05,15320,25,1
2,36.5,0.09,12581,20,1
3,6.2,0.49,11751,22,0
4,44.0,0.13,19140,14,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Non-Linear Position Dynamics: A simple rule like position <= 10 AND ctr < 1% treats a position 1 page with a 0.8% CTR identically to a position 9 page with a 0.8% CTR. However, position 1 expected CTR is vastly higher than position 9, meaning the true opportunity gap on position 1 is much larger.

Multi-Feature Interaction: Decay is driven by non-linear combinations of age, update frequency, position tiers, and search volume. Hardcoded if/else statements become unmanageable when attempting to balance all these variables simultaneously.

Ranked Prioritization: Fixed rules output binary True/False decisions, whereas an ML regressor outputs a continuous Opportunity Score that allows editorial teams to execute updates in exact order of ROI potential.

In [14]:
from sklearn.tree import DecisionTreeRegressor

# Demonstrate rule vs ML baseline setup
features = ["avg_position", "word_count", "days_since_last_update", "content_age_days"]
X = df[features].fillna(0)
y_actual = df["impressions_90d"].values

# Fit a simple tree regressor to learn expected impressions
reg = DecisionTreeRegressor(max_depth=4, random_state=42)
reg.fit(X, y_actual)

# Compute expected impressions and opportunity score gap
df["expected_impressions"] = reg.predict(X)
df["opportunity_score"] = df["expected_impressions"] - df["impressions_90d"]

# Calculate Precision@50 of the ML opportunity score
ml_precision = precision_at_k(df["opportunity_score"], df["is_declining"], k=50)

# Calculate Precision@50 of a naive rule (stale + high impression rule)
df["hand_rule"] = (df["days_since_last_update"] > 180).astype(int) * df["impressions_90d"]
rule_precision = precision_at_k(df["hand_rule"], df["is_declining"], k=50)

print(f"Hand Rule Precision@50:   {rule_precision:.3f}")
print(f"ML Opportunity Precision@50: {ml_precision:.3f}")

Hand Rule Precision@50:   0.680
ML Opportunity Precision@50: 0.740


In [15]:
from sklearn.model_selection import train_test_split

# 1. Split the data (75% for training, 25% held out for testing)
df_train, df_test = train_test_split(df, test_size=0.25, random_state=42)

# 2. Train the depth-4 model ONLY on the training data
reg_split = DecisionTreeRegressor(max_depth=4, random_state=42)
reg_split.fit(df_train[features].fillna(0), df_train["impressions_90d"])

# 3. Predict and evaluate ONLY on the unseen test data
df_test = df_test.copy() # Prevents pandas warnings
df_test["expected_impressions"] = reg_split.predict(df_test[features].fillna(0))
df_test["opportunity_score"] = df_test["expected_impressions"] - df_test["impressions_90d"]

# 4. Calculate the true, out-of-sample Precision@50
split_precision = precision_at_k(df_test["opportunity_score"], df_test["is_declining"], k=50)

print(f"ML Opportunity Precision@50 (Unseen Test Data): {split_precision:.3f}")

ML Opportunity Precision@50 (Unseen Test Data): 0.680


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.